# Running on Real Hardware

## Time to leave the simulator. This notebook covers the practical workflow of taking a Qiskit (or Cirq, or tket) circuit and running it on an actual quantum processor — choosing a backend, submitting jobs, retrieving results, and reading the data sheet to know what to expect.

# 1. The cloud providers

## Most quantum hardware today is accessible only via the cloud — direct lab access requires being a researcher at the same institution. The main routes:

## - **IBM Quantum**. Free tier (`Open` plan) gives queue access to ~127-qubit Heron systems. Pay-as-you-go for priority and the largest devices. SDK: **Qiskit**.
## - **IonQ**. Trapped-ion devices via the IonQ cloud or third-party platforms (AWS Braket, Azure Quantum, Google Cloud). SDK: their own + Qiskit/Cirq integrations.
## - **Quantinuum**. H-series ion traps via the Quantinuum API (commercial). SDKs: pytket, Qiskit, Cirq.
## - **AWS Braket**. Aggregator: access IonQ, Rigetti, IQM, QuEra, Oxford Quantum Circuits through one Python SDK.
## - **Azure Quantum**. Similar aggregator: IonQ, Quantinuum, Rigetti, PASQAL.
## - **Google Quantum AI**. Internal-only Sycamore/Willow access; **Cirq** is the public SDK but devices are not openly available.

## A typical learning path: start with **IBM Quantum's free tier** on the Qiskit ecosystem you already know. Move to **Braket** or **Azure** when comparing multiple modalities matters.

# 2. Setting up IBM Quantum

## ### One-time setup

## 1. Create an account at `quantum.ibm.com`.
## 2. Copy the API token from the dashboard.
## 3. Install the runtime client:

```bash
pip install qiskit qiskit-ibm-runtime
```

## 4. Save credentials locally (writes to `~/.qiskit/qiskit-ibm.json`):

```python
from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(channel='ibm_quantum', token='YOUR_TOKEN', overwrite=True)
```

## ### Listing available backends

```python
service = QiskitRuntimeService()
for backend in service.backends(simulator=False, operational=True):
    print(f'{backend.name:25s} qubits={backend.num_qubits:4d}  queue={backend.status().pending_jobs}')
```

## Pick the backend with the smallest queue and enough qubits. `service.least_busy(min_num_qubits=n)` automates this.

# 3. Submitting a job

## The modern Qiskit Runtime exposes two high-level **primitives**:

## - **`Sampler`** — runs a circuit and returns measurement outcomes (the histogram of bitstrings).
## - **`Estimator`** — runs a circuit + observable and returns $\langle\psi | O | \psi\rangle$ directly (no histograms).

## `Estimator` is what variational algorithms (VQE, QAOA) use: it handles measurement basis changes, shot allocation, and error mitigation internally.

```python
from qiskit import QuantumCircuit, transpile
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService()
backend = service.least_busy(min_num_qubits=2)

qc = QuantumCircuit(2, 2)
qc.h(0); qc.cx(0, 1); qc.measure([0, 1], [0, 1])
qc_t = transpile(qc, backend=backend, optimization_level=3)

sampler = SamplerV2(mode=backend)
job = sampler.run([qc_t], shots=4000)
print('Job ID:', job.job_id())
print('Job status:', job.status())
```

## Jobs go into a queue. Status transitions: `QUEUED` → `RUNNING` → `DONE`. Queue waits range from seconds (off-peak) to hours (busy chip). `job.result()` blocks until completion.

# 4. Reading results

```python
result = job.result()
pub_result = result[0]
counts = pub_result.data.c.get_counts()      # 'c' = name of the classical register
print(counts)
# {'00': 1986, '11': 1991, '01': 12, '10': 11}
```

## A noiseless Bell-state Sampler would give exactly $\{00, 11\}$ with 50/50 weights. The small `01` and `11` counts above are the noise — readout error plus two-qubit gate error. A useful sanity number is **fidelity**:

$$ \Large  F = P(00) + P(11) \approx 0.994. $$

## Anything north of 95% on a 2-qubit experiment is healthy on current hardware. A drop to 80% suggests a calibration drift on the chosen qubits — try a different pair (rerun layout/transpile) or different physical region.

# 5. Sessions, batches, and pricing

## The Runtime API has three job modes:

## - **Job mode.** One circuit submission per API call. Simplest.
## - **Batch mode.** Submit many jobs together; the queue treats them as a group, reducing per-job queue overhead.
## - **Session mode.** Reserve the backend for a window of (e.g.) 30 minutes. Within the session, your jobs jump the queue. Crucial for variational algorithms that submit thousands of small circuits sequentially.

```python
from qiskit_ibm_runtime import Session, EstimatorV2

with Session(backend=backend) as session:
    estimator = EstimatorV2(mode=session)
    for params in parameter_sweep:
        job = estimator.run([(qc, observable, params)])
        ...
```

## **Pricing.** IBM Open plan: ~10 minutes/month on free systems with rate limits. Pay-as-you-go: per-second billing of QPU time (around \$1.60/sec on the largest systems in 2025). Estimator/Sampler shots-per-second varies by backend and is documented in the data sheet.

# 6. Other SDK paths

## ### AWS Braket — one SDK, many devices

```bash
pip install amazon-braket-sdk
```

```python
from braket.aws import AwsDevice
from braket.circuits import Circuit

device = AwsDevice("arn:aws:braket:::device/qpu/ionq/Aria-1")    # trapped ions
circ = Circuit().h(0).cnot(0, 1)
task = device.run(circ, shots=1000)
print(task.result().measurement_counts)
```

## ### Cirq + Google Quantum AI (research access)

```python
import cirq, cirq_google
q0, q1 = cirq.LineQubit.range(2)
c = cirq.Circuit(cirq.H(q0), cirq.CNOT(q0, q1), cirq.measure(q0, q1))
sim = cirq.Simulator()
print(sim.run(c, repetitions=1000).histogram(key='q(0),q(1)'))
```

## ### pytket — backend-agnostic compiler with Qiskit/Cirq interop

```bash
pip install pytket-qiskit pytket-quantinuum
```

## pytket sometimes produces noticeably tighter circuits than Qiskit's native transpiler, especially for ion-trap devices.

# 7. A reproducible workflow

## A practical checklist before submitting any real job:

## 1. **Simulate first** with a noiseless backend to confirm the algorithm logic.
## 2. **Simulate again** with `AerSimulator.from_backend(real_backend)` — this builds a noise model from the device's calibration data. Catches issues before paying for QPU time.
## 3. **Transpile at optimization_level=3** for production runs.
## 4. **Inspect the transpiled circuit:** check two-qubit gate count, depth, and chosen physical qubits.
## 5. **Pick a good initial layout.** `mapomatic` or `qiskit.transpiler.passes.SabreLayout` automate this — they look at *current* calibration data to choose the lowest-error qubits.
## 6. **Use enough shots.** Standard error on a probability is $\sim 1/\sqrt{N_{\text{shots}}}$. For 1% precision: ~10000 shots.
## 7. **Apply measurement error mitigation** in post-processing.
## 8. **Save the job ID and backend snapshot.** Hardware changes daily; reproducibility requires recording which device version was used.

# Recap

## - Cloud providers: **IBM Quantum** (Qiskit), **Quantinuum** (pytket), **IonQ**, **AWS Braket**, **Azure Quantum**.
## - Two runtime primitives: **`Sampler`** (counts) and **`Estimator`** (expectation values, plus built-in mitigation).
## - **Session** mode is what variational algorithms need.
## - Always **simulate with the device's noise model** before paying for QPU time.
## - Reproducibility = record backend snapshot, calibration, and job ID.

## **End of chapter 8.** With qubit hardware, topology, noise, and the cloud workflow understood, the next chapter — quantum error correction — answers the natural question: given that real qubits are noisy, how do we build a reliable computer out of them?